In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import numpy as np
import os
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.cluster import KMeans

# ==================== 1. 数据增强 ====================
class AdvancedSimCLRTransform:
    def __init__(self, image_size=32, strong_aug=True):
        self.image_size = image_size
        self.strong_aug = strong_aug
        self.mean = [0.4914, 0.4822, 0.4465]
        self.std = [0.2470, 0.2435, 0.2616]
        
    def __call__(self, x):
        if self.strong_aug:
            transform1 = transforms.Compose([
                transforms.RandomResizedCrop(size=self.image_size, scale=(0.08, 1.0)),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomApply([transforms.ColorJitter(0.8,0.8,0.8,0.2)], p=0.8),
                transforms.RandomGrayscale(p=0.2),
                transforms.ToTensor(),
                transforms.Normalize(self.mean, self.std),
            ])
            transform2 = transforms.Compose([
                transforms.RandomResizedCrop(size=self.image_size, scale=(0.08, 1.0)),
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomApply([transforms.ColorJitter(0.4,0.4,0.4,0.1)], p=0.8),
                transforms.RandomGrayscale(p=0.2),
                transforms.ToTensor(),
                transforms.Normalize(self.mean, self.std),
            ])
        else:
            transform1 = transforms.Compose([
                transforms.RandomResizedCrop(self.image_size, scale=(0.2,1.0)),
                transforms.RandomHorizontalFlip(),
                transforms.ToTensor(),
                transforms.Normalize(self.mean, self.std),
            ])
            transform2 = transform1
        return transform1(x), transform2(x)

# ==================== 2. 模型架构 ====================
class ImprovedResNetBackbone(nn.Module):
    def __init__(self, base_encoder='resnet50', pretrained=False):
        super().__init__()
        if base_encoder == 'resnet18':
            encoder = torchvision.models.resnet18
            self.global_dim = 512
            self.local_dim = 256
        elif base_encoder == 'resnet50':
            encoder = torchvision.models.resnet50
            self.global_dim = 2048
            self.local_dim = 1024
        else:
            raise ValueError(f"Unsupported base_encoder: {base_encoder}")
        
        try:
            self.backbone = encoder(weights="DEFAULT") if pretrained else encoder(weights=None)
        except:
            self.backbone = encoder(pretrained=pretrained)
        
        self.backbone.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        self.backbone.maxpool = nn.Identity()
        self.backbone.fc = nn.Identity()
        
        self.layer1 = self.backbone.layer1
        self.layer2 = self.backbone.layer2
        self.layer3 = self.backbone.layer3
        self.layer4 = self.backbone.layer4
    
    def forward(self, x):
        x = self.backbone.conv1(x)
        x = self.backbone.bn1(x)
        x = self.backbone.relu(x)
        x = self.backbone.maxpool(x)
        
        x = self.layer1(x)
        x = self.layer2(x)
        x3 = self.layer3(x)
        x4 = self.layer4(x3)
        
        x_global = nn.functional.adaptive_avg_pool2d(x4, (1,1)).flatten(1)
        x_local = nn.functional.adaptive_avg_pool2d(x3, (4,4))
        return x_global, x_local

class ImprovedProjectionHead(nn.Module):
    def __init__(self, input_dim, output_dim=256, hidden_dim=2048):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, output_dim),
        )
    def forward(self, x):
        return self.mlp(x)

class ImprovedSimCLRModel(nn.Module):
    def __init__(self, backbone, global_projector, local_projector):
        super().__init__()
        self.encoder = backbone
        self.global_projector = global_projector
        self.local_projector = local_projector
    
    def forward(self, x):
        global_feat, local_feat = self.encoder(x)
        
        z_global = self.global_projector(global_feat)
        z_global = nn.functional.normalize(z_global, dim=1)
        
        local_feat = local_feat.flatten(1)
        z_local = self.local_projector(local_feat)
        z_local = nn.functional.normalize(z_local, dim=1)
        
        return z_global, z_local, global_feat

# ==================== 3. 聚类工具函数 ====================
def extract_features_for_clustering(model, dataloader, device='cuda'):
    model.eval()
    features_list = []
    with torch.no_grad():
        for images, _ in tqdm(dataloader, desc="Extracting features"):
            images = images.to(device)
            _, _, feat = model(images)
            features_list.append(feat.cpu())
    all_features = torch.cat(features_list, dim=0).numpy()
    return all_features

def perform_clustering(features, n_clusters=10):
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(features)
    print(f"Clusters: {len(np.unique(cluster_labels))}")
    return cluster_labels

class ClusteredDataset(torch.utils.data.Dataset):
    def __init__(self, dataset, cluster_labels, transform=None):
        self.dataset = dataset
        self.cluster_labels = cluster_labels
        self.transform = transform

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        img, _ = self.dataset[idx]
        if self.transform is not None:
            v1, v2 = self.transform(img)
        else:
            v1 = v2 = transforms.ToTensor()(img)
        cluster_label = self.cluster_labels[idx]
        return v1, v2, cluster_label

# ==================== 4. 聚类负样本损失（已修复）====================
def clustered_negative_simclr_loss(z1, z2, cluster_labels_batch, temperature=0.5):
    device = z1.device
    batch_size = z1.shape[0]
    pos_sim = torch.sum(z1 * z2, dim=1) / temperature
    z_full = torch.cat((z1, z2), dim=0)
    sim_matrix_full = torch.mm(z_full, z_full.T) / temperature

    pos_mask = torch.zeros((2 * batch_size, 2 * batch_size), dtype=torch.bool, device=device)
    pos_mask[torch.arange(batch_size), torch.arange(batch_size) + batch_size] = True
    pos_mask[torch.arange(batch_size) + batch_size, torch.arange(batch_size)] = True

    mask_neg = torch.ones((2 * batch_size, 2 * batch_size), dtype=torch.bool, device=device)
    mask_neg.fill_diagonal_(False)
    mask_neg[pos_mask] = False

    cl_labels_expanded = cluster_labels_batch.repeat(2)
    same_cluster_mask_full = (cl_labels_expanded.unsqueeze(0) == cl_labels_expanded.unsqueeze(1))
    mask_neg &= ~same_cluster_mask_full

    sim_matrix_masked = sim_matrix_full.masked_fill(~mask_neg, -65504.0)

    pos_sim_full = torch.cat([pos_sim, pos_sim], dim=0)
    exp_sim_neg = torch.exp(sim_matrix_masked).sum(dim=1)
    exp_pos_sim = torch.exp(pos_sim_full)
    
    log_prob_pos = pos_sim_full - torch.log(exp_pos_sim + exp_sim_neg + 1e-8)
    loss = -log_prob_pos.mean()

    return loss

# ==================== 5. 联合损失 ====================
def combined_clustered_loss(z1_g, z2_g, z1_l, z2_l, cluster_labels, temp=0.5, alpha=0.7, beta=0.3):
    loss_g = clustered_negative_simclr_loss(z1_g, z2_g, cluster_labels, temp)
    loss_l = clustered_negative_simclr_loss(z1_l, z2_l, cluster_labels, temp)
    total = alpha * loss_g + beta * loss_l
    return total, loss_g, loss_l

# ==================== 6. 训练 ====================
def train_epoch(model, loader, opt, device, epoch, alpha=0.7, beta=0.3, use_amp=True):
    model.train()
    total_loss = 0
    scaler = torch.cuda.amp.GradScaler() if use_amp else None
    
    pbar = tqdm(loader, desc=f"Epoch {epoch}")
    for v1, v2, cluster_y in pbar:
        v1, v2, cluster_y = v1.to(device), v2.to(device), cluster_y.to(device)
        opt.zero_grad()
        
        if use_amp:
            with torch.cuda.amp.autocast():
                z1_g, z1_l, _ = model(v1)
                z2_g, z2_l, _ = model(v2)
                loss, loss_g, loss_l = combined_clustered_loss(z1_g,z2_g,z1_l,z2_l, cluster_y, 0.5, alpha, beta)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
        else:
            z1_g, z1_l, _ = model(v1)
            z2_g, z2_l, _ = model(v2)
            loss, loss_g, loss_l = combined_clustered_loss(z1_g,z2_g,z1_l,z2_l, cluster_y, 0.5, alpha, beta)
            loss.backward()
            opt.step()
        
        total_loss += loss.item()
        pbar.set_postfix({"loss":f"{loss:.3f}", "g":f"{loss_g:.2f}", "l":f"{loss_l:.2f}"})
    return total_loss / len(loader)

# ==================== 7. 线性评估 ====================
def linear_evaluation_simple(model, train_loader, test_loader, num_classes=10, epochs=50, device='cuda'):
    model.eval()
    for p in model.parameters(): p.requires_grad = False
    
    linear = nn.Sequential(
        nn.Linear(2048, 512), nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(0.5),
        nn.Linear(512, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.3),
        nn.Linear(256, num_classes)
    ).to(device)
    
    opt = optim.AdamW(linear.parameters(), lr=0.01, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(opt, T_max=epochs)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    best_acc = 0
    
    for ep in range(epochs):
        linear.train()
        for x,y in train_loader:
            x,y = x.to(device), y.to(device)
            with torch.no_grad():
                _, _, feat = model(x)
            loss = criterion(linear(feat), y)
            opt.zero_grad(); loss.backward(); opt.step()
        
        linear.eval()
        cor, tot = 0,0
        with torch.no_grad():
            for x,y in test_loader:
                x,y = x.to(device), y.to(device)
                _, _, feat = model(x)
                cor += (linear(feat).argmax(1)==y).sum()
                tot += y.size(0)
        acc = 100.*cor/tot
        if acc>best_acc: best_acc=acc
        print(f"Linear Ep{ep+1} | Acc: {acc:.2f}% | Best: {best_acc:.2f}%")
        scheduler.step()
    return best_acc

# ==================== 主函数 ====================
def main():
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")
    
    config = {
        'batch_size': 128,
        'lr': 3e-4,
        'epochs': 50,
        'alpha': 0.7,
        'beta': 0.3,
        'base_encoder':'resnet50',
        'pretrained':True,
        'use_amp':True,
    }
    
    os.makedirs('checkpoints_comb', exist_ok=True)
    image_size = 32

    raw_train_ds = datasets.CIFAR10('./data', train=True, download=True)
    eval_tfm = transforms.Compose([transforms.ToTensor(), transforms.Normalize([0.4914,0.4822,0.4465],[0.2470,0.2435,0.2616])])
    feature_loader = DataLoader(datasets.CIFAR10('./data', train=True, transform=eval_tfm), batch_size=128, shuffle=False, num_workers=2)

    backbone = ImprovedResNetBackbone(config['base_encoder'], config['pretrained'])
    global_proj = ImprovedProjectionHead(backbone.global_dim, 256, 2048)
    local_proj = ImprovedProjectionHead(backbone.local_dim*4*4, 256, 2048)
    model = ImprovedSimCLRModel(backbone, global_proj, local_proj).to(device)

    print("\n=== Extract features： ===")
    features = extract_features_for_clustering(model, feature_loader, device)
    cluster_labels = perform_clustering(features, n_clusters=10)

    train_ds = ClusteredDataset(
        raw_train_ds,
        cluster_labels,
        transform=AdvancedSimCLRTransform(image_size, True)
    )
    train_loader = DataLoader(train_ds, config['batch_size'], shuffle=True, num_workers=2, pin_memory=True, drop_last=True)

    opt = optim.AdamW(model.parameters(), lr=config['lr'], weight_decay=1e-4)
    scheduler = CosineAnnealingLR(opt, T_max=config['epochs'])

    losses = []
    for ep in range(config['epochs']):
        loss = train_epoch(model, train_loader, opt, device, ep+1, config['alpha'], config['beta'], config['use_amp'])
        scheduler.step()
        losses.append(loss)
        print(f"Epoch {ep+1}/{config['epochs']} | Loss: {loss:.4f} | LR: {opt.param_groups[0]['lr']:.6f}")
    
    torch.save(model.state_dict(), 'checkpoints_comb/final_comb_model.pth')

    print("\n=== Start linear evaluation ===")
    linear_train = DataLoader(datasets.CIFAR10('./data',train=True,transform=eval_tfm), 128, shuffle=True, num_workers=2)
    linear_test = DataLoader(datasets.CIFAR10('./data',train=False,transform=eval_tfm), 128, shuffle=False, num_workers=2)
    best_acc = linear_evaluation_simple(model, linear_train, linear_test, 10, 50, device)
    
    print("\n" + "="*50)
    print(f"FINAL BEST ACCURACY: {best_acc:.2f}%")
    print("="*50)

if __name__ == "__main__":
    torch.manual_seed(42)
    np.random.seed(42)
    main()

Using device: cuda
Files already downloaded and verified

=== Extract features： ===


Extracting features: 100%|██████████| 391/391 [00:11<00:00, 33.00it/s]


Clusters: 10


Epoch 1: 100%|██████████| 390/390 [00:51<00:00,  7.57it/s, loss=4.119, g=4.11, l=4.15]


Epoch 1/50 | Loss: 4.3650 | LR: 0.000300


...

Epoch 48/50 | Loss: 3.8327 | LR: 0.000001


Epoch 49: 100%|██████████| 390/390 [00:50<00:00,  7.79it/s, loss=3.833, g=3.83, l=3.83]


Epoch 49/50 | Loss: 3.8329 | LR: 0.000000


Epoch 50: 100%|██████████| 390/390 [00:50<00:00,  7.77it/s, loss=3.827, g=3.82, l=3.84]


Epoch 50/50 | Loss: 3.8321 | LR: 0.000000

=== Start linear evaluation ===
Linear Ep1 | Acc: 84.33% | Best: 84.33%
Linear Ep2 | Acc: 85.60% | Best: 85.60%
Linear Ep3 | Acc: 85.95% | Best: 85.95%
Linear Ep4 | Acc: 86.25% | Best: 86.25%
Linear Ep5 | Acc: 86.08% | Best: 86.25%
Linear Ep6 | Acc: 86.46% | Best: 86.46%
Linear Ep7 | Acc: 86.71% | Best: 86.71%
Linear Ep8 | Acc: 86.63% | Best: 86.71%
Linear Ep9 | Acc: 87.04% | Best: 87.04%
Linear Ep10 | Acc: 87.20% | Best: 87.20%
Linear Ep11 | Acc: 87.20% | Best: 87.20%
Linear Ep12 | Acc: 87.48% | Best: 87.48%
Linear Ep13 | Acc: 87.62% | Best: 87.62%
Linear Ep14 | Acc: 87.39% | Best: 87.62%
Linear Ep15 | Acc: 87.71% | Best: 87.71%
Linear Ep16 | Acc: 88.01% | Best: 88.01%
Linear Ep17 | Acc: 87.91% | Best: 88.01%
Linear Ep18 | Acc: 88.33% | Best: 88.33%
Linear Ep19 | Acc: 88.56% | Best: 88.56%
Linear Ep20 | Acc: 88.08% | Best: 88.56%
Linear Ep21 | Acc: 88.18% | Best: 88.56%
Linear Ep22 | Acc: 88.70% | Best: 88.70%
Linear Ep23 | Acc: 88.52% | Best